# Week 38, Day 3: Complex cells and divisive normalisation

Companion notebook for the Engram lesson at https://engram-neuroai.vercel.app/learn/w038/d3.
Run the cells in order. Exercises have a checks cell after them: run it to test your answer.

### Exercise: Simple against complex

Write f1f0(r): the amplitude of the first Fourier harmonic of a response sampled over one cycle, 2 |F[1]| / N, divided by the mean |F[0]| / N. Then compute it for a simple cell (rectified even Gabor) and a complex cell (sum of squares of even and odd Gabors) as a grating drifts through 32 phases.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
n = 48; xy = (np.arange(n) - n / 2) * (4 / n)
X, Y = np.meshgrid(xy, xy)

def gabor(theta, f, phase=0.0, sx=0.4, sy=0.6):
    t = np.deg2rad(theta)
    xr = X * np.cos(t) + Y * np.sin(t); yr = -X * np.sin(t) + Y * np.cos(t)
    return np.exp(-xr ** 2 / (2 * sx ** 2) - yr ** 2 / (2 * sy ** 2)) * np.cos(2 * np.pi * f * xr + phase)

def grating(theta, f, phase, c=1.0):
    t = np.deg2rad(theta)
    return c * np.cos(2 * np.pi * f * (X * np.cos(t) + Y * np.sin(t)) + phase)

def f1f0(r):
    F = np.fft.rfft(r)
    return ...

phases = np.linspace(0, 2 * np.pi, 32, endpoint=False)
g_even, g_odd = gabor(0, 1.0), gabor(0, 1.0, np.pi / 2)
simple = np.array([max(0.0, np.sum(g_even * grating(0, 1.0, p))) for p in phases])
complex_ = np.array([np.sum(g_even * grating(0, 1.0, p)) ** 2 + np.sum(g_odd * grating(0, 1.0, p)) ** 2 for p in phases])
print(f"F1/F0: simple {f1f0(simple):.2f}, complex {f1f0(complex_):.3f}")
plt.plot(phases, simple / simple.max(), label="simple"); plt.plot(phases, complex_ / complex_.max(), label="complex")
plt.xlabel("grating phase (rad)"); plt.ylabel("normalised response"); plt.legend(); plt.show()

In [ ]:
import numpy as np
def test_ratio():
    "the simple cell's F1/F0 is above 1 and the complex cell's is below 0.1"
    assert f1f0(simple) > 1 and f1f0(complex_) < 0.1

def test_fn():
    "f1f0 of a pure half-wave rectified sinusoid is about 1.57"
    s = np.maximum(0, np.cos(np.linspace(0, 2 * np.pi, 64, endpoint=False)))
    assert abs(f1f0(s) - np.pi / 2) < 0.02

test_ratio()
test_fn()
print("All checks passed")

### Exercise: Saturation and contrast-invariant tuning

Write normalise(stim): the energies of complex cells at the 12 orientations in oris for this stimulus, divided by sigma2 plus their sum. Then compute the preferred cell's contrast response and the population tuning at 10% and 100% contrast.

In [ ]:
oris = np.arange(0, 180, 15)
bank = [(gabor(o, 1.0), gabor(o, 1.0, np.pi / 2)) for o in oris]
energy = lambda stim: np.array([np.sum(ge * stim) ** 2 + np.sum(go * stim) ** 2 for ge, go in bank])
sigma2 = 0.05 * energy(grating(0, 1.0, 0)).sum()          # half saturation near 22% contrast

def normalise(stim):
    E = energy(stim)
    return ...

contrasts = np.array([0.02, 0.05, 0.1, 0.2, 0.3, 0.5, 0.7, 1.0])
R_c = np.array([normalise(grating(0, 1.0, 0, c))[0] for c in contrasts])
tun_lo, tun_hi = normalise(grating(0, 1.0, 0, 0.1)), normalise(grating(0, 1.0, 0, 1.0))
# a threshold model for comparison: rectified energy above a fixed threshold
fine = np.arange(0, 180, 3)
E0 = lambda stim: np.sum(bank[0][0] * stim) ** 2 + np.sum(bank[0][1] * stim) ** 2
top = E0(grating(0, 1.0, 0))
thr = lambda c: np.maximum(0, np.array([E0(grating(o, 1.0, 0, c)) for o in fine]) - 0.02 * top)
thr_lo, thr_hi = thr(0.2), thr(1.0)
width = lambda t: np.sum(t >= 0.5 * t.max())
print("contrast response (relative to 100%):", {float(c): round(float(v), 2) for c, v in zip(contrasts, R_c / R_c.max())})
print("threshold model, orientations above half maximum: 20% contrast", width(thr_lo), " 100% contrast", width(thr_hi))
print("normalised tuning at 10% and 100%:", np.round(tun_lo / tun_lo.max(), 2), np.round(tun_hi / tun_hi.max(), 2))
plt.semilogx(contrasts, R_c, "o-"); plt.xlabel("contrast"); plt.ylabel("response"); plt.show()

In [ ]:
import numpy as np
def test_saturation():
    "the response at 100% contrast is less than 1.3 times the response at 50%"
    assert R_c[-1] < 1.3 * R_c[list(contrasts).index(0.5)]

def test_invariant():
    "normalised tuning curves at 10% and 100% contrast differ by less than 0.01 everywhere"
    assert np.max(np.abs(tun_lo / tun_lo.max() - tun_hi / tun_hi.max())) < 0.01

def test_threshold_broadens():
    "a threshold model's tuning is broader at high contrast than at low"
    assert width(thr_hi) > width(thr_lo)

test_saturation()
test_invariant()
test_threshold_broadens()
print("All checks passed")

### Exercise: Cross-orientation suppression

Compute the preferred (0 degree) cell's normalised response to the preferred grating at 50% contrast alone (r_test), to the orthogonal 90 degree grating at 50% alone (r_mask), and to their sum, a plaid (r_plaid).

In [ ]:
test = grating(0, 1.0, 0, 0.5); mask = grating(90, 1.0, 0, 0.5)
r_test = ...
r_mask = ...
r_plaid = ...
print(f"preferred alone {r_test:.3f}, mask alone {r_mask:.5f}, plaid {r_plaid:.3f} ({1 - r_plaid / r_test:.0%} suppression)")

In [ ]:
import numpy as np
def test_mask_alone():
    "the orthogonal mask alone barely drives the preferred cell"
    assert r_mask < 0.01 * r_test

def test_suppression():
    "adding the mask reduces the response to the preferred grating by at least 10%"
    assert r_plaid < 0.9 * r_test

test_mask_alone()
test_suppression()
print("All checks passed")